# 01. Molecule Visualization on Snowflake Notebooks

Snowflakeで抽出した候補薬剤の構造を、Snowsightの **Workspaces** で開いたNotebook（Run on Container）上で RDKit と py3Dmol を使って可視化します。

このNotebookは **Snowflake上で完結** します。ローカルにJupyter環境を用意する必要はなく、Workspaces から `.ipynb` を開いて実行できます。

## 前提

事前に以下が実行済みであることを前提にしています。

1. `sql/setup.sql` で `MULTIOMICS_HANDSON` データベースおよび `multiomics_drug_response_view` 等が作成されている
2. `sql/setup_notebook.sql` で以下が作成・付与されている
   - Compute Pool: `multiomics_compute_pool`
   - Role: `DATASCIENTIST`（必要権限付与済み）
3. Snowsightで以下の手順を実施している
   - 右上のロールを `DATASCIENTIST` に切り替える
   - 左メニュー `Workspaces` を開き、対象のワークスペースを選択
   - `+ Add new` → `Notebook (.ipynb)` で `01_molecule_visualization.ipynb` を作成（または同名ファイルをアップロード）
   - 上部の `Connect` → `Create new service` で以下を指定して Service を作成
     - Runtime: 最新のCPUランタイム（例: `v2.5 | CPU | Python 3.12`）
     - Compute pool: `multiomics_compute_pool`
     - Artifact repositories: `SNOWFLAKE.SNOWPARK.PYPI_SHARED_REPOSITORY`
     - External Access Integrations: なし（共有リポジトリで rdkit / py3Dmol が解決できる前提）

**注意**: Workspaces のNotebookでは、Connect時に Database / Schema / Warehouse は選びません。次のセルで `USE` 文を使って切り替えます。

## 注意

ここで表示している3D構造は、SMILESから生成した代表的なコンフォマーです。
実際の結晶構造やタンパク質結合状態を表すものではありません。
また、3D構造を眺めただけで薬効や結合親和性を判断することはできません。
実際の創薬研究では、標的タンパク質構造、結合ポケット、ドッキング、分子動力学、実験検証などが必要になります。

## パッケージのインストール

Container Runtime では Notebook内から直接 `pip install` できます。RDKitとpy3Dmolを入れます。

In [ ]:
!pip install --quiet rdkit py3Dmol

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw, AllChem
import py3Dmol

from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Workspaces の Notebook では接続時に DB/Schema/Warehouse を指定できないので
# ここで明示的に切り替える
session.sql("USE DATABASE MULTIOMICS_HANDSON").collect()
session.sql("USE SCHEMA   MULTIOMICS_HANDSON.PUBLIC").collect()
session.sql("USE WAREHOUSE HANDSON_WH").collect()

session.sql("SELECT CURRENT_DATABASE() AS db, CURRENT_SCHEMA() AS sch, CURRENT_WAREHOUSE() AS wh").to_pandas()

## Snowflakeから候補薬剤を直接取得する

`candidate_compounds.sql` の主要クエリ（Sensitive な組み合わせを SMILES と標的情報つきで）をそのまま発行します。

In [ ]:
candidates_df = session.sql(
    """
    SELECT
      v.sample_id,
      v.cell_line_name,
      v.cancer_type,
      v.gene_symbol,
      v.tpm,
      v.protein_abundance,
      v.drug_id,
      v.drug_name,
      v.ic50_um,
      v.response_label,
      c.chembl_id,
      c.pubchem_cid,
      c.canonical_smiles,
      t.target_symbol,
      t.target_name
    FROM multiomics_drug_response_view v
    JOIN compounds c
      ON v.drug_id = c.drug_id
    LEFT JOIN compound_targets t
      ON v.drug_id = t.drug_id
    WHERE v.response_label = 'Sensitive'
    ORDER BY v.ic50_um ASC
    """
).to_pandas()

candidates_df.head(20)

## 薬剤名ごとのユニークなSMILESを整理する

同じ薬剤が複数の細胞株に対してSensitiveになっているため、可視化用に薬剤単位でユニーク化します。

In [ ]:
unique_drugs_df = (
    candidates_df[["DRUG_NAME", "CANONICAL_SMILES", "CHEMBL_ID", "PUBCHEM_CID"]]
    .drop_duplicates(subset=["DRUG_NAME"])
    .reset_index(drop=True)
)

unique_drugs_df

## 2D構造を表示する

まずは1つ目の候補薬剤の2D構造を描画します。

In [ ]:
row    = unique_drugs_df.iloc[0]
name   = row["DRUG_NAME"]
smiles = row["CANONICAL_SMILES"]

print(f"{name}: {smiles}")

mol = Chem.MolFromSmiles(smiles)
Draw.MolToImage(mol, size=(500, 400), legend=name)

## 3Dコンフォマーを生成して py3Dmol で表示する

RDKitのETKDGv3で3D座標を埋め込み、MMFFで簡単に最適化したうえで、py3Dmolで描画します。

In [ ]:
mol3d = Chem.AddHs(mol)

params = AllChem.ETKDGv3()
params.randomSeed = 42

AllChem.EmbedMolecule(mol3d, params)
AllChem.MMFFOptimizeMolecule(mol3d)

mol_block = Chem.MolToMolBlock(mol3d)

viewer = py3Dmol.view(width=600, height=450)
viewer.addModel(mol_block, "mol")
viewer.setStyle({"stick": {}})
viewer.zoomTo()
viewer.show()

## 他の候補薬剤を切り替えて表示するヘルパー

`unique_drugs_df` 内の `DRUG_NAME` を渡せば、2Dと3Dをまとめて確認できます。

In [ ]:
def smiles_of(drug_name: str) -> str:
    matched = unique_drugs_df.loc[unique_drugs_df["DRUG_NAME"] == drug_name, "CANONICAL_SMILES"]
    if matched.empty:
        raise ValueError(f"{drug_name} is not in Sensitive candidates.")
    return matched.iloc[0]


def visualize_2d(drug_name: str):
    smiles = smiles_of(drug_name)
    mol    = Chem.MolFromSmiles(smiles)
    return Draw.MolToImage(mol, size=(500, 400), legend=drug_name)


def visualize_3d(drug_name: str, width: int = 600, height: int = 450):
    smiles = smiles_of(drug_name)
    mol    = Chem.MolFromSmiles(smiles)
    mol3d  = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = 42

    AllChem.EmbedMolecule(mol3d, params)
    AllChem.MMFFOptimizeMolecule(mol3d)

    mol_block = Chem.MolToMolBlock(mol3d)

    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(mol_block, "mol")
    viewer.setStyle({"stick": {}})
    viewer.zoomTo()
    return viewer.show()

In [ ]:
visualize_2d("Lapatinib")

In [ ]:
visualize_3d("Lapatinib")

In [ ]:
visualize_2d("Erlotinib")

In [ ]:
visualize_3d("Erlotinib")

## まとめ

- Snowsight の Workspaces から開いた `.ipynb`（Run on Container）の中で、Snowflakeに保存した候補薬剤のSMILESをそのままRDKit / py3Dmolで可視化できました。
- データを取り出してローカルに置く必要がないため、Snowflakeのアクセス制御や監査ログの中で再現性のある分析を行えます。
- 次のステップとして、Snowpark MLでの薬剤応答予測、Streamlit in Snowflakeでのダッシュボード化などへ自然に展開できます。